# 03 — Evaluation Analysis & Chapter 5 Figures
**Emotion-Aware E-Commerce Chatbot | MSc Dissertation | Heriot-Watt University**

This notebook:
1. Runs the full §5.4 evaluation pipeline
2. Loads results from `evaluation/results/`
3. Generates all Chapter 5 charts and tables

**Run `python evaluation/run_eval.py` first** to generate the JSON results file.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os, json, glob
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'Arial'
RESULTS_DIR = '../evaluation/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('✅ Imports ready')

In [ ]:
# ── Run evaluation now (or load existing results) ─────────────────────────────
import subprocess

# Check if results already exist
existing = sorted(glob.glob(f'{RESULTS_DIR}/eval_report_*.json'))
if existing:
    report_path = existing[-1]  # use most recent
    print(f'Using existing report: {report_path}')
else:
    print('No existing report found — running evaluation now...')
    result = subprocess.run(['python', '../evaluation/run_eval.py'],
                            capture_output=True, text=True)
    print(result.stdout)
    existing = sorted(glob.glob(f'{RESULTS_DIR}/eval_report_*.json'))
    report_path = existing[-1] if existing else None

if report_path:
    with open(report_path) as f:
        report = json.load(f)
    print(f'\n✅ Report loaded: {os.path.basename(report_path)}')
    print(f'   Sessions: {report["total_sessions"]} | Turns: {report["total_turns"]}')
else:
    print('⚠️  Could not find report — run: python evaluation/run_eval.py')

---
## §5.2 — System Performance Metrics

In [ ]:
# ── Response Latency Analysis ─────────────────────────────────────────────────
lat = report['system_performance']['response_latency']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of latency stats
metrics = ['Mean', 'Median', 'P95', 'Max']
values  = [lat['mean_ms'], lat['median_ms'], lat['p95_ms'], lat['max_ms']]
colors  = ['#1565C0','#1565C0','#E65100','#c62828']
bars = axes[0].bar(metrics, values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[0].axhline(1000, color='red', linestyle='--', linewidth=2, label='Target: <1000ms')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 f'{val:.0f}ms', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('§5.2 Response Latency', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Milliseconds')
axes[0].legend()

# Target met gauge
pct = lat['pct_under_1000ms']
axes[1].pie([pct, 100-pct], labels=[f'{pct:.0f}% under 1s', f'{100-pct:.0f}% over 1s'],
            colors=['#1B5E20','#ef5350'], autopct='%1.0f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title(f'Responses Meeting <1s Target\n{"✅ TARGET MET" if lat["meets_target"] else "⚠️ TARGET MISSED"}',
                  fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ch5_response_latency.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Avg: {lat["mean_ms"]}ms | Target met: {lat["meets_target"]}')

---
## §5.3 — Human-Centred Experience Metrics

In [ ]:
# ── Frustration Recovery Rate ─────────────────────────────────────────────────
frr = report['human_centred']['frustration_recovery_rate']
he  = report['human_centred']['handover_efficiency']
uss = report['human_centred']['user_satisfaction']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# FRR gauge
frr_pct = frr['pct']
axes[0].pie([frr_pct, 100-frr_pct],
            colors=['#1B5E20','#bdbdbd'],
            wedgeprops={'edgecolor':'white','linewidth':2},
            startangle=90)
axes[0].text(0, 0, f"{frr_pct:.0f}%", ha='center', va='center',
             fontsize=28, fontweight='bold', color='#1B5E20')
axes[0].set_title('Frustration Recovery Rate (FRR)\n§5.3', fontsize=12, fontweight='bold')

# Handover efficiency
axes[1].bar(['Handover\nRate', 'Avg Turn\nat Handover', 'Unnecessary\nHandovers'],
            [he['handover_pct'], he['avg_turn_at_handover']*10, he['unnecessary_pct']],
            color=['#1565C0','#E65100','#c62828'], edgecolor='white', linewidth=1.5, width=0.5)
axes[1].set_title('Handover Efficiency Metrics\n§5.3', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Value (Handover Rate & Unnecessary % are %)')
axes[1].text(0, he['handover_pct']+0.5, f"{he['handover_pct']:.0f}%", ha='center', fontweight='bold')
axes[1].text(1, he['avg_turn_at_handover']*10+0.5, f"T{he['avg_turn_at_handover']:.1f}", ha='center', fontweight='bold')
axes[1].text(2, he['unnecessary_pct']+0.5, f"{he['unnecessary_pct']:.0f}%", ha='center', fontweight='bold')

# Satisfaction scores
sat_keys   = ['satisfaction', 'perceived_empathy', 'trust_reliability']
sat_labels = ['Satisfaction', 'Perceived\nEmpathy', 'Trust &\nReliability']
sat_vals   = [uss[k]['mean'] for k in sat_keys]
sat_colors = ['#1565C0','#6A1B9A','#1B5E20']
bars = axes[2].bar(sat_labels, sat_vals, color=sat_colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[2].set_ylim(0, 5)
axes[2].axhline(3, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Neutral (3/5)')
axes[2].set_title('User Satisfaction Survey Scores\n§5.3 (1–5 Likert Scale)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Score (out of 5)')
for bar, val in zip(bars, sat_vals):
    if val:
        axes[2].text(bar.get_x()+bar.get_width()/2, val+0.05,
                     f'{val:.2f}/5', ha='center', fontweight='bold', fontsize=11)
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ch5_human_centred_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## §5.4 — Scenario-by-Scenario Analysis

In [ ]:
# ── Run scenario-level evaluation and plot frustration curves ─────────────────
from evaluation.run_eval import SCENARIOS, run_scenario, collect_survey_scores

scenario_sessions = {}
for name, scenario in SCENARIOS.items():
    print(f'Running: {name}')
    s = run_scenario(name, scenario['messages'])
    s = collect_survey_scores(s)
    scenario_sessions[name] = s

print('\n✅ All 3 scenarios complete')

In [ ]:
# ── Plot frustration trajectory per scenario ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
scenario_titles = {
    'product_query':    'Scenario 1: Product Query',
    'refund_complaint': 'Scenario 2: Refund Complaint',
    'delivery_complaint':'Scenario 3: Delivery Complaint'
}
colors = ['#1565C0','#BF360C','#1B5E20']

for ax, (name, session), color in zip(axes, scenario_sessions.items(), colors):
    turns  = [t.turn_number for t in session.turns]
    fscores = [t.frustration_score for t in session.turns]
    modes  = [t.response_mode for t in session.turns]

    ax.plot(turns, fscores, 'o-', color=color, linewidth=2.5, markersize=8, zorder=3)
    ax.fill_between(turns, fscores, alpha=0.15, color=color)

    # Threshold lines
    ax.axhline(0.5, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Alert (0.5)')
    ax.axhline(0.7, color='red',    linestyle='--', linewidth=1.5, alpha=0.7, label='Handover (0.7)')

    # Mark response modes
    for t, f, m in zip(turns, fscores, modes):
        if 'handover' in m:
            ax.annotate('🚨', (t, f), textcoords='offset points', xytext=(0, 10),
                        ha='center', fontsize=14)
        elif 'high' in m:
            ax.annotate('💛', (t, f), textcoords='offset points', xytext=(0, 10),
                        ha='center', fontsize=12)

    # Handover marker
    if session.handover_triggered and session.handover_turn:
        ax.axvline(session.handover_turn, color='red', linestyle=':',
                   linewidth=2, label=f'Handover T{session.handover_turn}')

    ax.set_title(scenario_titles[name], fontsize=12, fontweight='bold')
    ax.set_xlabel('Turn')
    ax.set_ylabel('Frustration Score')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('§5.4 — Frustration Trajectory by Scenario', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ch5_scenario_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Chapter 5 Summary Table ───────────────────────────────────────────────────
frr = report['human_centred']['frustration_recovery_rate']
he  = report['human_centred']['handover_efficiency']
uss = report['human_centred']['user_satisfaction']
lat = report['system_performance']['response_latency']

summary = pd.DataFrame([
    {'Metric': 'Frustration Recovery Rate (FRR)',    'Value': f"{frr['pct']:.0f}%",
     'Section': '§5.3', 'Target': '>60%', 'Status': '✅' if frr['pct']>=60 else '⚠️'},
    {'Metric': 'Handover Rate',                      'Value': f"{he['handover_pct']:.0f}%",
     'Section': '§5.3', 'Target': 'Contextual', 'Status': 'ℹ️'},
    {'Metric': 'Avg Turn at Handover',               'Value': f"{he['avg_turn_at_handover']:.1f}",
     'Section': '§5.3', 'Target': '3-6 turns', 'Status': '✅'},
    {'Metric': 'Unnecessary Handovers',              'Value': f"{he['unnecessary_pct']:.0f}%",
     'Section': '§5.3', 'Target': '<20%', 'Status': '✅' if he['unnecessary_pct']<20 else '⚠️'},
    {'Metric': 'User Satisfaction Score',            'Value': f"{uss['satisfaction']['mean']}/5",
     'Section': '§5.3', 'Target': '>3.5/5', 'Status': '✅' if uss['satisfaction']['mean'] and uss['satisfaction']['mean']>=3.5 else '⚠️'},
    {'Metric': 'Perceived Empathy Score',            'Value': f"{uss['perceived_empathy']['mean']}/5",
     'Section': '§5.3', 'Target': '>3.5/5', 'Status': '✅' if uss['perceived_empathy']['mean'] and uss['perceived_empathy']['mean']>=3.5 else '⚠️'},
    {'Metric': 'Mean Response Latency',              'Value': f"{lat['mean_ms']:.0f}ms",
     'Section': '§5.2', 'Target': '<1000ms', 'Status': '✅' if lat['mean_ms']<1000 else '⚠️'},
    {'Metric': '95th Percentile Latency',            'Value': f"{lat['p95_ms']:.0f}ms",
     'Section': '§5.2', 'Target': '<1000ms', 'Status': '✅' if lat['p95_ms']<1000 else '⚠️'},
])

print('\n===  CHAPTER 5 RESULTS SUMMARY  ===')
display(summary)
summary.to_csv(f'{RESULTS_DIR}/ch5_summary_table.csv', index=False)
print(f'\n✅ All charts and tables saved to {RESULTS_DIR}/')
print('   Use these directly in your dissertation Chapter 5.')